In [ ]:
using JuMP
using SumOfSquares
using DynamicPolynomials
using SCS

## 1. Using JULIA and the SumOfSquaresLibrary to check that f is *not* 2-sos-submodular

In [ ]:
n = 5 #size of ground set
@polyvar x[1:n] #create variable x

F=3*x[1]*x[2]*x[3]*x[4]*x[5]-4*x[3]*x[1]*x[2]-9*x[4]*x[1]*x[2]-12*x[1]*x[2]*x[4]*x[5]-4*x[1]*x[2]*x[3]*x[5]-4*x[1]*x[2]*x[3]*x[4]+2 #MLE of f
t=2 #for t-sos-submodular

model = SOSModel(SCS.Optimizer) #instantiates the model

#builds the set I2[x] 
expr = :( $(x[1]) * (1 - $(x[1])) == 0 )
for i in 2:n
    expr = :($expr && $(x[i]) * (1 - $(x[i])) == 0)
end
S2=eval(:(@set $expr))

#builds all second derivatives
second_derivatives = Matrix{Any}(undef, n, n)
for i in 1:n, j in 1:n
    second_derivatives[i, j] = differentiate(differentiate(F, x[i]), x[j])
    @constraint(model, -second_derivatives[i, j] >= 0, domain=S2, maxdegree=2*t) #requires that -\partial^2 F/partial xi xi be t-sos modulo I2[x]
end

optimize!(model) #solves
st = termination_status(model) #returns the termination status
st
st_example1_t2 = st


As can be seen above, the problem is infeasible, which means that $F$ is not 2-sos-submodular.

## 2. Using JULIA and the SumOfSquaresLibrary to check that f *is* 3-sos-submodular 

In [ ]:
n = 5 #size of ground set
@polyvar x[1:n] #create variable x

F=3*x[1]*x[2]*x[3]*x[4]*x[5]-4*x[3]*x[1]*x[2]-9*x[4]*x[1]*x[2]-12*x[1]*x[2]*x[4]*x[5]-4*x[1]*x[2]*x[3]*x[5]-4*x[1]*x[2]*x[3]*x[4]+2 #MLE of f
t=3 #for t-sos-submodular

model = SOSModel(SCS.Optimizer) #instantiates the model

#builds the set I2[x] 
expr = :( $(x[1]) * (1 - $(x[1])) == 0 )
for i in 2:n
    expr = :($expr && $(x[i]) * (1 - $(x[i])) == 0)
end
S2=eval(:(@set $expr))

#builds all second derivatives
second_derivatives = Matrix{Any}(undef, n, n)
for i in 1:n, j in 1:n
    second_derivatives[i, j] = differentiate(differentiate(F, x[i]), x[j])
    @constraint(model, -second_derivatives[i, j] >= 0, domain=S2, maxdegree=2*t) #requires that -\partial^2 F/partial xi xi be t-sos modulo I2[x]
end

optimize!(model) #solves
st = termination_status(model) #returns the termination status
st
st_example1_t3 = st


As can be seen above, the problem is feasible, which implies that $F$ is 3-sos-submodular and thus submodular.

In [ ]:
import MathOptInterface as MOI

@assert st_example1_t2 in (MOI.INFEASIBLE, MOI.ALMOST_INFEASIBLE)
@assert st_example1_t3 in (MOI.OPTIMAL, MOI.ALMOST_OPTIMAL)

println("Final check passed: Example 1 gives infeasible for t=2 and feasible for t=3, as stated in the draft.")
